In [2]:
import sys
print(sys.executable)
print(sys.version)

/home/stefan_stringer/Deep_Learning_Project/Project/.venv/bin/python
3.13.0 (main, Nov 13 2024, 19:57:38) [GCC 11.4.0]


In [12]:
!pip install --upgrade mani_skill torch

Defaulting to user installation because normal site-packages is not writeable
  Using cached mani_skill-3.0.1-py3-none-any.whl.metadata (9.3 kB)
  Using cached torch-2.15.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (36 kB)
  Using cached dacite-1.9.2-py3-none-any.whl.metadata (17 kB)
  Using cached h5py-3.16.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached gitpython-3.2.1-py3-none-any.whl.metadata (13 kB)
  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
  Using cached transforms3d-0.4.2-py3-none-any.whl.metadata (2.8 kB)
  Using cached trimesh-5.2.1-py3-none-any.whl.metadata (13 kB)
  Using cached imageio-2.38.2-py3-none-any.whl.metadata (9.7 kB)
ERROR: Could not find a version that satisfies the requirement mplib==0.1.1 (from mani-skill) (from versions: 0.2.0a0, 0.2.0a1, 0.2.0a2, 0.2.0, 0.2.1)
ERROR: No matching distribution found for mplib==0.1.1 (from mani-skill)


In [13]:
#Should be able to run now - but the video doesn't seem to work
!python -m mani_skill.examples.demo_random_action -e PushCube-v1 \
  --render-mode="rgb_array" --record-dir="videos"

/home/stefan_stringer/Deep_Learning_Project/Project/.venv/bin/python: Error while finding module specification for 'mani_skill.examples.demo_random_action' (ModuleNotFoundError: No module named 'mani_skill')


In [14]:
!pip install lerobot
!pip install -e ".[smolvla]"

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Obtaining file:///home/stefan_stringer/Deep_Learning_Project/torch_vla_implementation
ERROR: file:///home/stefan_stringer/Deep_Learning_Project/torch_vla_implementation does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [15]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base")

ModuleNotFoundError: No module named 'lerobot'

In [9]:
for n, p in policy.named_parameters():   # inspecting the insane structure behind the model
    print(n, p.shape, p.requires_grad) 

NameError: name 'policy' is not defined

In [15]:
print("total params:", sum(p.numel() for p in policy.parameters()) / 1000000, "mil")
print("trainable params:", sum(p.numel() for p in policy.parameters() if p.requires_grad) / 1000000, "mil")

total params: 450.046176 mil
trainable params: 99.880992 mil


In [ ]:
# LION optimiser as found via: https://github.com/google/automl/blob/master/lion/lion_pytorch.py
# Write up about it found here: https://aiwiki.ai/wiki/lion_optimizer perks is that it is a memory efficient optimiser that is perfect for smolVLA
import torch
from torch.optim import Optimizer as Optimiser

class LION(Optimiser):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0):
        """Initialize the hyperparameters.

    Args:
      params (iterable): iterable of parameters to optimize or dicts defining
        parameter groups
      lr (float, optional): learning rate (default: 1e-4)
      betas (Tuple[float, float], optional): coefficients used for computing
        running averages of gradient and its square (default: (0.9, 0.99))
      weight_decay (float, optional): weight decay coefficient (default: 0)
    """
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                
                p.data.mul_(1 - group['lr'] * group['weight_decay'])
                
                grad = p.grad
                state = self.state[p]
                
                # State initialisation
                if len(state) == 0:
                # Exponential moving average of gradient values
                    state['exp_avg'] = torch.zeros_like(p)

                exp_avg = state['exp_avg']
                beta1, beta2 = group['betas']
                
                update = exp_avg * beta1 + grad * (1 - beta1)
                p.add_(update.sign(), alpha=-group['lr'])
                
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)
            
            return loss

In [ ]:
import torch

lr_AW = 1e-4
lr_lion = 2e-5 #documentation suggests that LION should be used with a lower learning rate than AdamW
weight_decay = 0.01

optimiser_AW = torch.optim.AdamW([p.numel() for p in policy.parameters() if p.requires_grad], lr = lr_AW, weight_decay = weight_decay) #Question about [p.numel() for p in policy.parameters() if p.requires_grad]
#for n, p in policy.named_parameters():   # inspecting the insane structure behind the model
#    print(n, p.shape, p.requires_grad) 
#Why is it not structured like this? so  trainable_params = [p for n, p in policy.named_parameters() if p.requires_grad] and then optimiser_AW = torch.optim.AdamW(trainable_params, lr = lr_AW, weight_decay = weight_decay)  - I think this is the right way to do it
# optimiser_LION = LION([p.numel() for p in policy.parameters() if p.requires_grad], lr = lr_lion) #IF this is how we get trainable parameters

#for batch in train_loader:
    #here we can have the usual training loop 
    #we deffo need some checkpoints here though 

In [7]:
import gymnasium as gym
import mani_skill.envs

env = gym.make(
    "PushCube-v1", # there are more tasks e.g. "PickCube-v1", "PegInsertionSide-v1", ...
    num_envs=1,
    obs_mode="state", # there is also "state_dict", "rgbd", ...
    control_mode="pd_ee_delta_pose", # there is also "pd_joint_delta_pos", ...
    render_mode = "rgb_array"
)
print("Observation space", env.observation_space)
print("Action space", env.action_space)

obs, _ = env.reset(seed=0) # reset with a seed for determinism
done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    env.render()  # a display is required to render
env.close()

Observation space Box(-inf, inf, (1, 35), float32)
Action space Box(-1.0, 1.0, (7,), float32)


In [ ]:
import gymnasium as gym
import torch
from collections import defaultdict
from mani_skill.vector.wrappers.gymnasium import ManiSkillVectorEnv
env_id = "PushCube-v1"
num_eval_envs = 64
env_kwargs = dict(obs_mode="state") # modify your env_kwargs here
eval_envs = gym.make(env_id, num_envs=num_eval_envs, reconfiguration_freq=1, **env_kwargs)
# add any other wrappers here
eval_envs = ManiSkillVectorEnv(eval_envs, ignore_terminations=True, record_metrics=True)

# evaluation loop, which will record metrics for complete episodes only
obs, _ = eval_envs.reset(seed=0)
eval_metrics = defaultdict(list)
for _ in range(400):
    action = eval_envs.action_space.sample() # replace with your policy action
    obs, rew, terminated, truncated, info = eval_envs.step(action)
    # note as there are no partial resets, truncated is True for all environments at the same time
    if truncated.any():
        for k, v in info["final_info"]["episode"].items():
            eval_metrics[k].append(v.float())
for k in eval_metrics.keys():
    print(f"{k}_mean: {torch.mean(torch.stack(eval_metrics[k])).item()}")
    